In [1]:
import pandas as pd
import numpy as np
import re

In [2]:
df = pd.read_csv(r"C:\Users\Junayed\pandas_prac\Aug_9\messy_tickets.csv")

In [3]:
df.shape

(42, 16)

In [4]:
df.dtypes

TicketID             str
CustomerName         str
Email                str
Product              str
Category             str
Priority             str
Status               str
CreatedDate          str
ResolvedDate         str
ResolutionTime       str
AgentName            str
Channel              str
SatisfactionScore    str
UpdatedAt            str
Region               str
Notes                str
dtype: object

In [5]:
df.isna().sum()

TicketID              0
CustomerName          0
Email                 0
Product               0
Category              0
Priority              0
Status                0
CreatedDate           0
ResolvedDate         10
ResolutionTime       11
AgentName             0
Channel               0
SatisfactionScore     7
UpdatedAt             0
Region                0
Notes                37
dtype: int64

In [6]:
df.head(5)

,TicketID,CustomerName,Email,Product,Category,Priority,Status,CreatedDate,ResolvedDate,ResolutionTime,AgentName,Channel,SatisfactionScore,UpdatedAt,Region,Notes
0,TK5001,JosÃ© GarcÃ­a,jose.garcia@gmail.com,Router X200,Networking,High,Closed,2023-07-01 09:00,2023-07-01 14:00,5h,Dana Kim,Email,4,2023-07-01 14:05,West,NaN
1,TK5002,Amara Chukwu,amara.chukwu@gmail.com,Modem Z50,Networking,Medium,Closed,2023-07-01 10:00,2023-07-02 09:00,1d 5h,Leo Park,Phone,5,2023-07-02 09:10,East,NaN
2,TK5003,FranÃ§ois Dubois,francois.dubois@gmail.com,Router X200,Networking,Low,Open,2023-07-01 11:00,NaN,NaN,Dana Kim,Chat,NaN,2023-07-01 11:00,West,NaN
3,TK5004,Priya Nair,priya.nair@gmail.com,Smart Hub,Smart Home,High,Closed,2023-07-02 08:00,2023-07-02 10:30,2h 30m,Leo Park,email,3,2023-07-02 10:35,east,NaN
4,TK5005,JosÃ© GarcÃ­a,jose.garcia@gmail.com,Router X200,Networking,High,Closed,2023-07-01 09:00,2023-07-01 14:00,5h,Dana Kim,Email,4,2023-07-01 14:02,West,NaN


In [7]:
df.tail(6)

,TicketID,CustomerName,Email,Product,Category,Priority,Status,CreatedDate,ResolvedDate,ResolutionTime,AgentName,Channel,SatisfactionScore,UpdatedAt,Region,Notes
36,TK5037,Femi Adeyemi,femi.adeyemi@gmail.com,Modem Z50,Networking,Medium,Closed,2023-07-12 10:00,2023-07-13 07:00,21h,Leo Park,Chat,4,2023-07-13 07:05,North,NaN
37,TK5038,Katarina Novak,katarina.novak@gmail.com,Router X200,Networking,High,Closed,2023-07-13 08:00,2023-07-13 10:00,2h,Dana Kim,Email,5,2023-07-13 10:05,West,NaN
38,TK5039,Hugo Bernard,hugo.bernard@gmail.com,Smart Hub,Smart Home,Medium,Closed,2023-07-13 09:00,2023-07-13 19:00,10h,Rita Alves,Phone,4,2023-07-13 19:05,South,NaN
39,TK5040,Layla Haddad,layla.haddad@gmail.com,Modem Z50,networking,Low,Closed,2023-07-13 10:00,2023-07-13 15:00,5h,Leo Park,Chat,5,2023-07-13 15:05,North,NaN
40,TK5041,Connor Doyle,connor.doyle@gmail.com,Router X200,Networking,High,In Progress,2023-07-14 08:00,NaN,NaN,Dana Kim,Email,unknown,2023-07-14 08:00,West,NaN
41,TK5042,Yara Mansour,yara.mansour@gmail.com,Smart Hub,Smart Home,Medium,Closed,2023-07-14 09:00,2023-07-14 10:00,1h,Rita Alves,Phone,3,2023-07-14 10:05,South,NaN


## Fixing Character Encoding Artifacts (Mojibake)

**Issue Identified:**  
The `CustomerName` column contains character encoding artifacts (Mojibake), where non-ASCII characters (e.g., `é`, `í`, `ç`, `ø`) were double-encoded as UTF-8 string sequences (e.g., `JosÃ© GarcÃ­a` instead of `José García`).

**Fix Applied:**  
1. **Byte Decoding:** Encoded the strings back to `latin-1` (ISO-8859-1) bytes to restore raw byte streams.
2. **UTF-8 Reconstruction:** Decoded the bytes using `utf-8` to restore special characters to their proper representations (e.g., `Françoise`, `Luís`, `Bjørn`).

In [8]:
chk = "JosÃ© GarcÃ­a"
chk.encode("latin1").decode("utf-8")

'José García'

In [9]:
df["CustomerName"] = df["CustomerName"].str.encode("latin1").str.decode("utf-8")
df["CustomerName"]

0         José García
1        Amara Chukwu
2     François Dubois
3          Priya Nair
4         José García
5        Mateus Silva
6      Ingrid Solberg
7         Chidi Okoro
8         Naledi Dube
9         Anders Berg
10      Wanjiru Kamau
11       Luís Pereira
12        Grace Owusu
13        Tariq Malik
14      Elin Fransson
15       Mateus Silva
16         Sana Malik
17       Bjørn Haugen
18        Noor Haddad
19       Kwame Mensah
20        Yuki Tanaka
21     Camille Girard
22    Diego Fernandez
23    Isabela Cardoso
24       Otto Kessler
25          Aditi Rao
26       Youssef Amin
27       Lena Fischer
28      Marco Villani
29        Hana Suzuki
30         Ravi Patel
31           Nia Osei
32      Tobias Kruger
33        Aisha Bello
34        Bianca Ruiz
35         Owen Marsh
36       Femi Adeyemi
37     Katarina Novak
38       Hugo Bernard
39       Layla Haddad
40       Connor Doyle
41       Yara Mansour
Name: CustomerName, dtype: str

## String Title Casing Normalization

**Issue Identified:**  
Text columns (`Status`, `Priority`, and `Category`) contain inconsistent character casing (e.g., lowercased values like `"networking"` and `"smart home"` mixed with `"Networking"` and `"Smart Home"`), which impacts grouping and visual consistency.

**Fix Applied:**  
1. **Title Casing Applied:** Used `.str.title()` across target text fields to capitalize the first letter of each word consistently.
2. **Data Consistency Enforced:** Standardized values to ensure uniform categorical grouping (e.g., normalizing `"smart home"` to `"Smart Home"`).

In [10]:
df["Product"] = df["Product"].astype(str).str.strip().str.title()
df["Category"] = df["Category"].astype(str).str.strip().str.title()
df["Priority"] = df["Priority"].astype(str).str.strip().str.title()
df["Status"] = df["Status"].astype(str).str.strip().str.title()

df[["Product", "Category", "Priority", "Status"]]

,Product,Category,Priority,Status
0,Router X200,Networking,High,Closed
1,Modem Z50,Networking,Medium,Closed
2,Router X200,Networking,Low,Open
3,Smart Hub,Smart Home,High,Closed
4,Router X200,Networking,High,Closed
5,Modem Z50,Networking,Medium,In Progress
6,Smart Hub,Smart Home,Low,Closed
7,Router X200,Networking,High,Closed
8,Smart Hub,Smart Home,Medium,Closed
9,Modem Z50,Networking,Low,Open


## Standardizing CreatedDate and ResolvedDate Timestamps

**Issue Identified:**  
The `CreatedDate` and `ResolvedDate` columns contain timestamp entries stored as string objects (`object` dtype), preventing temporal analysis, duration calculations (e.g., time-to-resolution), and chronological sorting.

**Fix Applied:**  
1. **Parsed String Timestamps:** Applied `pd.to_datetime()` across both timestamp columns to convert string date-time representations into native Pandas `datetime64[ns]` objects.
2. **Standardized ISO Parsing:** Utilized `format='mixed'` and `errors='coerce'` to handle variations in date-time strings and safely manage missing/null resolution timestamps.

In [12]:
df["CreatedDate"] = pd.to_datetime(df["CreatedDate"], format = 'mixed', errors = 'coerce')
df["ResolvedDate"] = pd.to_datetime(df["ResolvedDate"], format = 'mixed', errors = 'coerce')

df[["CreatedDate", "ResolvedDate"]]

,CreatedDate,ResolvedDate
0,2023-07-01 09:00:00,2023-07-01 14:00:00
1,2023-07-01 10:00:00,2023-07-02 09:00:00
2,2023-07-01 11:00:00,NaT
3,2023-07-02 08:00:00,2023-07-02 10:30:00
4,2023-07-01 09:00:00,2023-07-01 14:00:00
5,2023-07-02 13:00:00,NaT
6,2023-07-03 09:00:00,2023-07-03 09:45:00
7,2023-07-03 10:00:00,NaT
8,2023-07-03 11:00:00,2023-07-04 11:00:00
9,2023-07-04 08:00:00,2023-07-03 20:00:00


**Now there might some orders duration might not be valid.**

In [13]:
resolved_before_created = df["ResolvedDate"] < df["CreatedDate"]

df.loc[resolved_before_created, ["TicketID", "CustomerName", "CreatedDate", "ResolvedDate"]]

,TicketID,CustomerName,CreatedDate,ResolvedDate
9,TK5010,Anders Berg,2023-07-04 08:00:00,2023-07-03 20:00:00


**TK5010:** resolved a full 12 hours before it was created which is logically impossible, Create a new column where it is stated that it needs a review.

In [16]:
df["NeedsReview"] = resolved_before_created

**Now all the `Status` valid?** There might be some ticket are still open but marked as closed and some may closed but marked as open.

In [22]:
mismatch_status = (
    ((df["Status"] == "Closed") & df["ResolvedDate"].isna()) | 
    ((df["Status"] == "Open") & df["ResolvedDate"].notna()) |
    ((df["Status"] == "In Progress") & df["ResolvedDate"].notna())
)

df.loc[mismatch_status, ["TicketID", "Status", "CreatedDate", "ResolvedDate", "Notes"]]

,TicketID,Status,CreatedDate,ResolvedDate,Notes
7,TK5008,Closed,2023-07-03 10:00:00,NaT,Missing resolved date
9,TK5010,Open,2023-07-04 08:00:00,2023-07-03 20:00:00,Resolved before created?


**TK5008** has been closed but there is no `ResolvedDate`, as there is no way to add the resolvedate then it needs reviews & **TK5010** also resolved before creating the ticket itself but `Status` is still open that means the ticket is still running then now we have remove the faulty ResolvedDate. 

In [28]:
df.loc[df["ResolvedDate"].dt.date == pd.to_datetime("2023-07-03").date(), "ResolvedDate"] = pd.NaT
df.loc[df["Notes"] == "Resolved before created?", "Notes"] = np.nan

In [35]:
mismatch_status = (
    ((df["Status"] == "Closed") & df["ResolvedDate"].isna()) | 
    (df["Status"].isin(["Open", "In Progress"]) & df["ResolvedDate"].notna())
)

In [36]:

df.loc[mismatch_status, ["TicketID", "Status", "CreatedDate", "ResolvedDate", "Notes"]]

,TicketID,Status,CreatedDate,ResolvedDate,Notes
6,TK5007,Closed,2023-07-03 09:00:00,NaT,NaN
7,TK5008,Closed,2023-07-03 10:00:00,NaT,Missing resolved date
15,TK5016,Closed,2023-07-02 13:00:00,NaT,Updated after resync
